# Matching pursuit on the folded rotation correlation

Greedy peak extraction (kernel-subtraction / matching pursuit):

1. find the **best kernel match**: the location whose *expected peak shape* (the
   descriptor autocorrelation `K`) has the largest fitted projection in the residual
2. fit its **amplitude** by least squares:  A = <r, K_mu> / <K_mu, K_mu>
3. **subtract** A*K(mu) from the residual
4. repeat until the remaining maximum falls below `STOP_FRAC` of the original max
   (default 50 %, exactly the idea: peaks below half the strongest correlation are
   not interesting)
5. final **joint non-negative re-fit** of all collected components

Because we work on the folded curve [0, pi), a component at mu automatically covers
mu and mu+pi; the circular kernel handles seam peaks (the 0/pi boundary).

All angles in **radians**.

In [ ]:
"""Imports + data location."""
import os
import numpy as np

# Data directory: this notebook lives in plotting_results/2d, debug output in ./data
DATA_DIR = os.path.join(os.getcwd(), "data")
if not os.path.isdir(DATA_DIR):
    DATA_DIR = "/home/tim-external/ros_ws/src/fsregistration/plotting_results/2d/data"
print("Data dir:", DATA_DIR)

"""Robust loaders (tolerate headers, tab/comma/space delimiters, trailing non-numeric tokens)."""

def load_lines(fname):
    """All parsed rows (list of lists of floats); headers / non-numeric rows skipped."""
    path = os.path.join(DATA_DIR, fname)
    if not os.path.isfile(path):
        return None
    out = []
    with open(path) as f:
        for ln in f:
            ln = ln.strip()
            if not ln or ln.startswith("#"):
                continue
            vals = []
            for t in ln.replace(",", " ").split():
                try:
                    vals.append(float(t))
                except ValueError:
                    break
            if vals:
                out.append(vals)
    return out

def load_csv(fname):
    """2D array using the dominant column count (e.g. registration_meta keeps its 13 numeric cols)."""
    rows = load_lines(fname)
    if not rows:
        return None
    counts = [len(r) for r in rows]
    modal = max(set(counts), key=counts.count)
    rows = [r for r in rows if len(r) == modal]
    return np.array(rows) if rows else None

def status(name, arr):
    print("  [%s] %s%s" % ("ok" if arr is not None else "MISSING", name,
                           "  (%d rows)" % arr.shape[0] if arr is not None and arr.ndim == 2 else ""))

In [ ]:
"""Load everything and print a summary (angles in rad)."""
print("Loading debug files from", DATA_DIR)
curve = load_csv("rotationCorrelation1D.csv")   # [index, angle(rad), normalizedCorrelation]
peaks = load_csv("rotationPeaks.csv")           # [angle, peakCorrelation, covariance, levelPotential, index]
meta  = load_csv("registration_meta.csv")       # frame1 frame2 rot_angle_deg tx ty ...
cfg   = load_lines("dataForReadIn.csv")         # parameter rows of mixed width

status("rotationCorrelation1D.csv", curve)
status("rotationPeaks.csv", peaks)
status("registration_meta.csv", meta)
import time as _t
for _f in ("rotationCorrelation1D.csv", "rotationPeaks.csv", "registration_meta.csv"):
    _p = os.path.join(DATA_DIR, _f)
    if os.path.isfile(_p):
        print("    mtime %s: %s" % (_f, _t.strftime("%Y-%m-%d %H:%M:%S", _t.localtime(os.path.getmtime(_p)))))

# --- parameters (dataForReadIn: 6-col metadata row + per-angle 3-col rows) ---
N = num_angles = num_total = None
level_thresh = None
if cfg:
    for r in cfg:
        if len(r) == 6:
            N = int(r[0]); _cn = int(r[1]); _cell = r[2]
            level_thresh = r[3]; num_angles = int(r[4]); num_total = int(r[5])
            break
print("\\nParameters: N=%s  level_potential=%s  numAngles=%s  numTotalSolutions=%s" % (N, level_thresh, num_angles, num_total))

# --- ground truth (derived: GT = estimated rotation + GT error), converted to rad ---
gt_rad, pair = None, None
est_rad = gt_err_rad = gt_minus_rad = None
if meta is not None and len(meta) >= 1:
    pair = (int(meta[0, 0]), int(meta[0, 1]))
    est_deg = float(meta[0, 2])
    gt_err_deg = float(meta[0, 7]) if meta.shape[1] > 7 else 0.0
    est_rad = np.deg2rad(est_deg)
    gt_err_rad = np.deg2rad(gt_err_deg)
    gt_rad = est_rad + gt_err_rad
    gt_minus_rad = est_rad - gt_err_rad          # alternative sign convention: error = est - true
    print("Pair: %d -> %d   estimated rotation: %.4f rad   GT error: %.4f rad   -> GT(est+err): %.4f rad"
          % (pair[0], pair[1], est_rad, gt_err_rad, gt_rad))
else:
    print("No registration_meta.csv -> ground truth marker will not be shown.")

# --- detected peaks ---
if peaks is not None:
    print("\\nDetected rotation peaks (from rotationPeaks.csv):")
    print("  %10s %11s %13s" % ("angle(rad)", "correlation", "levelPotential"))
    for p in peaks:
        print("  %10.4f %11.4f %13.4f" % (p[0], p[1], p[3]))
    print("  Note: every peak has an antipodal copy at +pi rad (the curve is pi-periodic).")
    print("  Physical rotations are angles modulo pi rad.")

In [ ]:
"""Fold the curve onto [0, pi) and build the kernel K (descriptor autocorrelation)."""
import plotly.graph_objects as go

th, F = None, None
if curve is None:
    print("No correlation curve available.")
else:
    x, c = curve[:, 1], curve[:, 2]

    # robust fold: uniform [0, 2pi) grid with an even number of samples
    n = len(x)
    step = np.median(np.diff(x))
    uniform = np.allclose(np.diff(x), step, atol=1e-9)
    if not uniform or not np.isclose(x[0], 0.0, atol=1e-6):
        xg = np.linspace(0.0, 2 * np.pi, n)
        c = np.interp(xg, x, c)
        x = xg
    n = len(x) - (len(x) % 2)
    x, c = x[:n], c[:n]
    half = n // 2
    th, F = x[:half], (c[:half] + c[half:]) / 2.0
    print("Folded grid: %d samples on [0, %.4f], step %.5f rad" % (len(th), th[-1], th[1] - th[0]))

# --- true kernel from the SH coefficients of the reference (scan 2) descriptor ---
c2R = load_csv("patCoefR_1angle.csv")
c2I = load_csv("patCoefI_1angle.csv")
c1R = load_csv("sigCoefR_1angle.csv")
c1I = load_csv("sigCoefI_1angle.csv")

def build_kernel_acf(cR, cI, theta):
    """Autocorrelation kernel of a descriptor from its SH coefficients:
    K(theta) = sum_{m!=0} (sum_l |c_lm|^2) cos(m theta).
    The coefficient array uses the alm layout of softRegistrationClass.cpp
    (see the almIdx formula there); the file is a raw dump of that array."""
    bw = int(round(np.sqrt(len(cR))))
    bigL = bw - 1
    Q = np.zeros(2 * bw)
    for l in range(bw):
        for m in range(-l, l + 1):
            if m >= 0:
                idx = m * (bigL + 1) - m * (m - 1) // 2 + (l - m)
            else:
                idx = bigL * (bigL + 3) // 2 + 1 + (bigL + m) * (bigL + m + 1) // 2 + (l - abs(m))
            Q[m + bw] += cR[idx] ** 2 + cI[idx] ** 2
    mpos = np.arange(1, bw)                       # Q is symmetric in m (descriptor is real)
    K = 2.0 * (Q[mpos + bw] @ np.cos(np.outer(mpos, theta)))
    return K

def build_kernel_empirical(theta):
    """Fallback: measure the peak shape from the strongest peak of the observed
    folded curve itself (mirrored, 0.9 rad wide)."""
    Fk = F
    i0 = int(np.argmax(Fk))
    W = int(round(0.9 / (theta[1] - theta[0])))
    seg = Fk[max(0, i0 - W):i0 + W + 1]
    K = np.zeros(len(theta))
    for j in range(W + 1):
        v = (seg[W - j] + (seg[W + j] if W + j < len(seg) else 0.0)) / 2.0
        K[j] = v
        if j > 0:
            K[len(theta) - j - 1] = v
    return K

if curve is None:
    print("Kernel panel skipped: no correlation curve.")
    K = None
elif c2R is not None and c2I is not None:
    print("Building true kernel K from scan-2 descriptor (patCoef, %d coefficients)" % len(c2R))
    K = build_kernel_acf(c2R[:, 0], c2I[:, 0], x)
    K = (K - K.min()) / (K.max() - K.min())
elif c1R is not None and c1I is not None:
    print("WARNING: patCoef missing -> using scan-1 descriptor (sigCoef) as kernel.")
    K = build_kernel_acf(c1R[:, 0], c1I[:, 0], x)
    K = (K - K.min()) / (K.max() - K.min())
else:
    print("WARNING: no coefficient files -> measuring the kernel empirically from the dominant peak.")
    K = build_kernel_empirical(x)
    K = (K - K.min()) / (K.max() - K.min())
_nk = len(K)
print("  K normalized to [0, 1]:  K(0) = %.3f  K(pi/4) = %.3f  K(pi/2) = %.3f  K(pi) = %.3f"
      % (K[0], K[_nk // 8], K[_nk // 4], K[_nk // 2]))

In [ ]:
"""Matching-pursuit core: greedy kernel subtraction on the folded curve (rad).

Each step: find the location with the largest fitted projection amplitude
A = <r, K_mu> / <K_mu, K_mu> (least squares), refine it locally, subtract A*K(mu)."""
FINE_RAD = 0.0004        # local refinement step (~0.02 deg)
COARSE_RAD = 0.01        # initial scan grid step (~0.57 deg)
MIN_SEP_RAD = 0.1        # min separation between components (~5.7 deg)
STOP_FRAC = 0.5          # stop when remaining max < 50% of the original max
DIA_FRAC = 0.25          # diagnostic run threshold: what a stricter cut would add
N_MAX = 6                # hard cap on the number of components

def circ_dist(a, b, period=np.pi):
    d = np.abs(np.asarray(a) - b) % period
    return np.minimum(d, period - d)

def kval(a):
    w = np.mod(np.asarray(a), 2 * np.pi)
    return np.interp(np.minimum(w, 2 * np.pi - w), x, K)   # K is even on [0, 2pi)

def proj_amp(r, a):
    """Least-squares amplitude of kernel K centered at a in the residual r."""
    Km = kval(th - a)
    den = float(np.sum(Km ** 2))
    return float(np.dot(r, Km) / den) if den > 0 else 0.0

def mp_run(stop_frac=STOP_FRAC, nmax=N_MAX, refine=True):
    """Greedy kernel subtraction. Returns (steps, r_final):
    steps = [(mu, A, peak_height_before, resid_max_after), ...]."""
    r = F.copy()
    r0max = float(r.max())
    stop = stop_frac * r0max
    steps = []
    grid = np.arange(COARSE_RAD, np.pi - COARSE_RAD, COARSE_RAD)
    for _it in range(nmax):
        As = np.array([proj_amp(r, a) for a in grid])
        for m, _A, _h, _rm in steps:                # suppress near-duplicates
            As[circ_dist(grid, m) < MIN_SEP_RAD] = -1.0
        i = int(np.argmax(As))
        if As[i] <= 0.0:
            break
        mu, A = float(grid[i]), float(As[i])
        if refine:                                   # local position refinement
            for off in np.arange(-0.05, 0.0501, FINE_RAD):
                a = (mu + off) % np.pi
                Aa = proj_amp(r, a)
                if Aa > A:
                    mu, A = float(a), float(Aa)
        h0 = float(r[np.argmin(np.abs(th - mu))])    # residual height right at mu
        r = r - A * kval(th - mu)
        steps.append((mu, A, h0, float(r.max())))
        if r.max() < stop:
            break
    return steps, r

def design_matrix(angles, th):
    return np.column_stack([kval(th - a) for a in angles] + [np.ones(len(th))])

def fit_nonneg_nb(th, Fw, angles_in):
    """Least squares, non-negative amplitudes, NO baseline (matching-pursuit semantics)."""
    kept = list(angles_in)
    while True:
        A = np.column_stack([kval(th - a) for a in kept])
        coef, res, *_ = np.linalg.lstsq(A, Fw, rcond=None)
        r = res[0] if len(res) else float(np.sum((A @ coef - Fw) ** 2))
        if len(kept) == 0 or np.all(coef >= 0):
            return coef, r, kept
        del kept[int(np.argmin(coef))]

def fit_nonneg(th, Fw, angles_in):
    """Least squares with non-negative component amplitudes, affine baseline.
    Returns (coef, r, kept_angles); negative-amplitude components are dropped."""
    kept = list(angles_in)
    while True:
        A = design_matrix(kept, th)
        coef, res, *_ = np.linalg.lstsq(A, Fw, rcond=None)
        r = res[0] if len(res) else float(np.sum((A @ coef - Fw) ** 2))
        amps = coef[:-1]
        if len(amps) == 0 or np.all(amps >= 0):
            return coef, r, kept
        del kept[int(np.argmin(amps))]

In [ ]:
"""Run the greedy extraction (50% stop + 25% diagnostic) and validate vs GT."""
# fixed palette for iteration coloring (no matplotlib dependency)
PALETTE = ["#d62728", "#1f77b4", "#2ca02c", "#9467bd", "#8c564b", "#e377c2"]

def fmt_row(it, mu, A, h0, rmax, r0max):
    return "  %d   %9.4f   %6.3f   %6.3f   %7.4f  %5.1f%%" % (it, mu, A, h0, rmax, 100.0 * rmax / r0max)

steps, r_fin = mp_run(STOP_FRAC, N_MAX)
steps2, r_fin2 = mp_run(DIA_FRAC, N_MAX)

if not steps:
    print("No kernel match with positive amplitude found - nothing to subtract.")
else:
    r0max = float(F.max())
    print("Matching pursuit on the folded correlation (stop < %.0f%% of original max %.4f):"
          % (100 * STOP_FRAC, r0max))
    print("  it     mu (rad)       A      peak    resid after    % of orig")
    for it, (mu, A, h0, rmax) in enumerate(steps):
        print(fmt_row(it, mu, A, h0, rmax, r0max))

    # joint non-negative re-fit of all collected components (fixes overlap errors)
    mus_all = [s[0] for s in steps]
    coef, r_j, kept = fit_nonneg_nb(th, F, mus_all)
    amp_j = {m: float(a) for m, a in zip(kept, coef)} if kept else {}
    print("\\nJoint non-negative re-fit of %d component(s):" % len(mus_all))
    for it, (mu, A, _h, _rm) in enumerate(steps):
        print("  %d: mu = %.4f rad  greedy A = %.3f  ->  joint A = %.3f%s"
              % (it, mu, A, amp_j.get(mu, 0.0),
                 "  (kept)" if mu in amp_j else "  (dropped)"))
    model = np.column_stack([kval(th - a) for a in kept]) @ coef
    rj_max = float(np.max(np.abs(F - model))) if kept else float(np.max(np.abs(F)))
    print("  joint residual (max |F - model|): %.4f  (greedy residual range [%.3f, %.3f])"
          % (rj_max, r_fin.min(), r_fin.max()))

    # diagnostic: what a stricter threshold would add
    extra = steps2[len(steps):] if len(steps2) > len(steps) else []
    if extra:
        print("\\nDiagnostic run at %.0f%% threshold - additional component(s):" % (100 * DIA_FRAC))
        for it, (mu, A, h0, rmax) in enumerate(extra, start=len(steps)):
            print(fmt_row(it, mu, A, h0, rmax, r0max))
    else:
        print("\\nDiagnostic run at %.0f%% threshold: no additional components." % (100 * DIA_FRAC))

    # GT validation (both sign conventions; the writing pipeline decides which is real)
    if gt_rad is not None:
        print("\\nGT validation (error sign convention depends on the writing pipeline):")
        print("    GT(est+err) = %.4f rad, GT(est-err) = %.4f rad" % (gt_rad, gt_minus_rad))
        for it, (mu, A, _h, _rm) in enumerate(steps):
            dp = circ_dist(mu, np.mod(gt_rad, np.pi))
            dm = circ_dist(mu, np.mod(gt_minus_rad, np.pi))
            d = min(dp, dm)
            conv = "est+err" if dp <= dm else "est-err"
            tag = "CORRECT" if d < 0.035 else ("near" if d < 0.1 else "far")
            print("    it %d: mu = %.4f rad -> %.4f rad (~%.2f deg) from nearer GT (%s)  [%s]"
                  % (it, mu, d, np.rad2deg(d), conv, tag))

    # comparison with classical persistence detection
    if peaks is not None:
        pfold = sorted(set(np.round(np.mod(peaks[:, 0], np.pi), 4)))
        print("\\nClassical persistence peaks (folded mod pi): %s" % ["%.4f" % p for p in pfold])
        print("    matching pursuit found:                 %s"
              % ", ".join("%.4f (A=%.3f)" % (mu, A) for mu, A, _h, _rm in steps))

In [ ]:
"""Figure 1: folded correlation + matching-pursuit components (star = antipodal copy)."""
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=th, y=F, name="folded correlation C (mod pi)",
                          line=dict(color="steelblue", width=2),
                          hovertemplate="%{x:.4f} rad<br>corr %{y:.4f}<extra></extra>"))
if peaks is not None:
    fig1.add_trace(go.Scatter(x=np.mod(peaks[:, 0], np.pi), y=peaks[:, 1], mode="markers",
                              name="persistence peaks (mod pi)",
                              marker=dict(symbol="x", size=11, color="red", line=dict(width=2)),
                              hovertemplate="%{x:.4f} rad<extra></extra>"))
cols = PALETTE
for it, (mu, A, _h, _rm) in enumerate(steps):
    star = mu + np.pi if mu + np.pi < 2 * np.pi else mu - np.pi + 2 * np.pi
    fig1.add_trace(go.Scatter(x=[mu], y=[A], mode="markers+text",
                              name="MP %d: %.4f rad (A=%.3f)" % (it + 1, mu, A),
                              marker=dict(symbol="diamond", size=11, color=cols[it],
                                          line=dict(width=1, color="black")),
                              text=["%d" % (it + 1)], textposition="bottom center",
                              textfont=dict(size=11, color=cols[it]),
                              hovertemplate="mu=%{x:.4f} rad, A={A:.3f}<extra></extra>"))
    fig1.add_trace(go.Scatter(x=[star], y=[A], mode="markers",
                              name="antipodal of %d" % (it + 1),
                              marker=dict(symbol="diamond-open", size=11, color=cols[it],
                                          line=dict(width=1.5, color=cols[it])),
                              showlegend=False,
                              hovertemplate="mu+pi=%{x:.4f} rad<extra></extra>"))
for g, nm in ((gt_rad, "GT(est+err)"), (gt_minus_rad, "GT(est-err)")):
    if g is not None:
        fig1.add_vline(x=float(np.mod(g, np.pi)), line=dict(color="green", dash="dash", width=1),
                       annotation_text="%s %.4f" % (nm, g), annotation_position="top left")
fig1.update_layout(
    title="Matching pursuit on the folded correlation, pair %d -> %d" % pair if pair else
          "Matching pursuit on the folded correlation",
    height=520, legend=dict(orientation="h", y=1.12, font=dict(size=10)),
    xaxis=dict(title="rotation angle (rad)", range=[0, np.pi],
               tickvals=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4, np.pi],
               ticktext=["0", "pi/4", "pi/2", "3pi/4", "pi"]),
    yaxis=dict(title="normalized correlation", range=[min(float(F.min()), 0) - 0.05, 1.05]),
    margin=dict(t=80))
fig1.show()
print("Figure 1: diamonds = matching-pursuit components (numbered, color = order),")
print("          open diamonds = antipodal copies at mu+pi, red x = persistence peaks.")

In [ ]:
"""Figure 2: the residual after each subtraction step (grey = original fold)."""
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=th, y=F, name="folded correlation (original)",
                          line=dict(color="grey", width=1.5, dash="dot")))
for it, (mu, A, _h, rmax) in enumerate(steps):
    r_tmp = F.copy()
    for mu2, A2, _h2, _rm2 in steps[:it + 1]:
        r_tmp = r_tmp - A2 * kval(th - mu2)
    fig2.add_trace(go.Scatter(x=th, y=r_tmp,
                              name="after %d subtraction(s)" % (it + 1),
                              line=dict(color=cols[it], width=2),
                              hovertemplate="%{x:.4f} rad<br>resid %{y:.4f}<extra></extra>"))
if steps:
    fig2.add_trace(go.Scatter(x=th, y=r_fin, name="final residual",
                              line=dict(color="black", width=2.5)))
fig2.update_layout(
    title="Residual after each greedy subtraction (pair %d -> %d)" % pair if pair else
          "Residual after each greedy subtraction",
    height=460, legend=dict(orientation="h", y=1.12, font=dict(size=10)),
    xaxis=dict(title="rotation angle (rad)", range=[0, np.pi],
               tickvals=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4, np.pi],
               ticktext=["0", "pi/4", "pi/2", "3pi/4", "pi"]),
    yaxis=dict(title="residual correlation"),
    margin=dict(t=80))
fig2.show()

# verdict
if steps:
    kmax = float(F.max())
    n50 = len(steps)
    print("\\nVerdict: %d component(s) at the %.0f%% threshold; final residual in [%.4f, %.4f] (~%.1f%% of the original %.4f)."
          % (n50, 100 * STOP_FRAC, r_fin.min(), r_fin.max(), 100.0 * r_fin.max() / kmax, kmax))
    print("  (negative residual = mild oversubtraction: the fitted kernel is broader than the local peak;")
    print("  The greedy subtraction stops when nothing above the threshold remains;")
    print("  the translation stage picks the final rotation from the candidates above.")